# Which loss: cross-entropy, soft labels or CORN?

**The question:** with the model fixed, does an ordinal loss beat
cross-entropy, for our baseline ABMIL and for the RoPE model, on mixed and
across hospitals?

ISUP grades are ordered: predicting 4 for a true 5 is a smaller error than
predicting 0, and quadratic weighted kappa scores it that way, but
cross-entropy treats every wrong grade alike. CLAUDE.md asks for the loss to
be settled before any hyperparameter tuning. The two candidates are the ones
Abdul Haleem compared in the FRSG repo (issues #9, #28, #35), defined exactly
as he did:

| Loss | Outputs | Training target | Predicted grade |
|---|---|---|---|
| cross-entropy (notebooks 12 and 15) | 6 | the true grade | highest logit |
| soft labels | 6 | a Gaussian over the grades centred on the true one, sigma 1.0 | highest logit |
| CORN (Shi, Cao and Raschka 2021) | 5 | "grade > k" for each threshold k, counted only once the previous threshold is passed | number of cumulative probabilities above 0.5 |

His finding (EfficientNet on Radboud patches, two seeds, #28): soft labels
0.722, ahead of CORAL (0.696, unstable between seeds) and cross-entropy (0.649);
a later test (#35) found CORN behind soft labels too. Whether that holds
for frozen UNI2-h features with an attention head, and across hospitals, is
what this notebook measures.

Both heads are carried (decision 2026-11-18): the baseline `GatedABMIL` and the
RoPE model `TileTransformerABMIL(rope=True)`. Everything else is fixed: the
packed tissue set, the recipe, the protocols and seeds, the paired bootstrap
and the decision rule. Within each head, each loss is compared with that
head's cross-entropy run.

## Preamble

In [1]:
"""Make the project's src/ folder importable.

Locally it sits one folder up from notebooks/. On Kaggle it arrives as an
attached dataset, so every attached dataset is checked for a src/ folder, and
the notebook stops with a clear message if there is none.
"""

import glob
import os
import sys

SRC_ROOT = next(
    (
        root
        for root in [os.path.abspath("..")] + sorted(glob.glob("/kaggle/input/*"))
        if os.path.isfile(os.path.join(root, "src", "tiling.py"))
    ),
    None,
)
if SRC_ROOT is None:
    raise FileNotFoundError("src/ not found; on Kaggle, attach the project's src folder as a dataset")
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

In [2]:
"""Imports.

functools, os, time   building each head's model, paths, timing.
pandas                tables.
tqdm                  progress over folds and seeds.
src.features          the packed set, and where predictions are saved.
src.mil               the heads, the losses, protocols, saved predictions and comparisons.
src.paths             where folds.csv is, locally and on Kaggle.
"""

import functools
import os
import time

import pandas as pd
from tqdm.auto import tqdm

from src.features import (
    open_packed,  # the packed set: features memory-mapped, index, starts, ends, tiles table
    predictions_file,  # where a model's predictions on a set are saved
)
from src.mil import (
    CROSS_SITE,  # the two cross-site directions: name -> (training hospital, test hospital)
    GatedABMIL,  # the baseline head
    TileTransformerABMIL,  # the RoPE head (rope=True)
    compare_runs,  # two saved prediction tables of one protocol: QWKs, difference, paired bootstrap interval
    cross_site,  # every slide of one hospital to every slide of the other, once per seed
    cross_validate,  # the mixed protocol on the Stage 4 folds
    grade_distribution,  # share of slides per grade, true against predicted
    load_bags,  # every slide's features into RAM
    load_positions,  # every slide's tile grid positions, for the RoPE head
    prediction_columns,  # the prediction columns of a saved table
    save_predictions,  # predictions to <version>/predictions/<name>.parquet
    seed_summary,  # QWK per seed and their mean and spread
)
from src.paths import (
    DATA_DIR,  # folds.csv lives here locally
    find_input,  # finds folds.csv
)

In [3]:
"""Settings and labels.

1. HEADS: name -> (model builder, whether it needs positions). The names match
   the cross-entropy predictions already saved: "abmil" (notebook 12) and
   "roformer" (notebook 15).
2. NEW_LOSSES are trained here; cross-entropy is read from those saved runs.
   A run is saved as "<head>_<loss>-<protocol>".
3. The run stops if a cross-entropy run is missing.
"""

VERSION = "uni2h-L1-224-t144-hsv-v1"
HEADS = {
    "abmil": (GatedABMIL, False),
    "roformer": (functools.partial(TileTransformerABMIL, rope=True), True),
}
NEW_LOSSES = ["soft_labels", "corn"]
PROTOCOLS = ["mixed", *CROSS_SITE]


def run_name(head, loss):
    return head if loss == "cross_entropy" else f"{head}_{loss}"


packed = open_packed(VERSION)
folds = pd.read_csv(find_input("folds.csv", local_dir=DATA_DIR)).set_index("slide_id")
labels = folds.loc[packed.index["slide_id"]]
missing = [f"{h}-{p}" for h in HEADS for p in PROTOCOLS if not os.path.exists(predictions_file(VERSION, f"{h}-{p}"))]
if missing:
    raise FileNotFoundError(f"cross-entropy predictions missing: {missing}; run notebooks 12 and 15 first")
print(f"{VERSION}: {len(labels):,} slides; heads {list(HEADS)}; new losses {NEW_LOSSES}")

folds.csv: /home/aztrek/projects/Prostate-Cancer-Grading/data/folds.csv
uni2h-L1-224-t144-hsv-v1: 10,615 slides; heads ['abmil', 'roformer']; new losses ['soft_labels', 'corn']


## Train and predict

Each head with each new loss, under the three protocols. About 25 minutes per
loss for ABMIL and 100 for the RoPE model: about 4 hours in all.

In [4]:
"""Train and predict every head with every new loss under every protocol; save the predictions."""

bags = load_bags(packed)
positions = load_positions(packed)
for head, (model_fn, needs_positions) in HEADS.items():
    for loss in NEW_LOSSES:
        started = time.perf_counter()
        extra = {"model_fn": model_fn, "positions": positions if needs_positions else None, "loss": loss}
        predicted = {"mixed": cross_validate(bags, labels, "mixed", progress=tqdm, **extra)}
        for name, (train_hospital, test_hospital) in CROSS_SITE.items():
            predicted[name] = cross_site(bags, labels, train_hospital, test_hospital, progress=tqdm, **extra)
        for protocol, values in predicted.items():
            save_predictions(VERSION, f"{run_name(head, loss)}-{protocol}", labels, values)
        print(f"{head} with {loss}: trained in {(time.perf_counter() - started) / 60:.1f} min")

abmil with soft_labels: trained in 26.7 min
abmil with corn: trained in 27.0 min
roformer with soft_labels: trained in 102.9 min
roformer with corn: trained in 100.2 min


## Scores

Every head and loss per protocol (seed mean and range for cross-site), then each
new loss against its head's cross-entropy run with paired bootstrap intervals.

In [5]:
"""Score every head and loss, then compare each new loss with the same head's cross-entropy."""


def table(name, protocol):
    return pd.read_parquet(predictions_file(VERSION, f"{name}-{protocol}"))


scores = []
for head in HEADS:
    for loss in ["cross_entropy", *NEW_LOSSES]:
        row = {"head": head, "loss": loss}
        for protocol in PROTOCOLS:
            t = table(run_name(head, loss), protocol)
            s = seed_summary(t["isup_grade"], t[prediction_columns(t)])
            row[protocol] = s["mean"]
            if protocol != "mixed":
                row[f"{protocol} range"] = f"{s['min']:.3f} to {s['max']:.3f}"
        scores.append(row)
print(pd.DataFrame(scores).round(4).to_string(index=False))

rows = []
for head in HEADS:
    for loss in NEW_LOSSES:
        for protocol in PROTOCOLS:
            for row in compare_runs(table(head, protocol), table(run_name(head, loss), protocol), protocol):
                rows.append({"head": head, "loss": loss, **row})
comparison = pd.DataFrame(rows)
print()
print(comparison.round(4).to_string(index=False))

    head          loss  mixed  radboud_to_karolinska radboud_to_karolinska range  karolinska_to_radboud karolinska_to_radboud range
   abmil cross_entropy 0.8643                 0.6316              0.602 to 0.656                 0.7800              0.775 to 0.784
   abmil   soft_labels 0.8858                 0.6963              0.678 to 0.709                 0.7886              0.785 to 0.793
   abmil          corn 0.8717                 0.7125              0.702 to 0.723                 0.7905              0.789 to 0.792
roformer cross_entropy 0.8740                 0.7189              0.653 to 0.744                 0.7660              0.753 to 0.777
roformer   soft_labels 0.8870                 0.7501              0.722 to 0.780                 0.7937              0.788 to 0.801
roformer          corn 0.8764                 0.7333              0.708 to 0.748                 0.7774              0.759 to 0.791

    head        loss              protocol     slides     n  first  second 

## Predicted grades

Ordinal losses should move errors towards neighbouring grades. Share of
slides per predicted grade, seeds pooled, for each cross-site direction.

In [6]:
"""Predicted-grade distribution per cross-site direction: true, then every head and loss."""

for protocol in CROSS_SITE:
    base = table("abmil", protocol)
    shares = {"true": grade_distribution(base["isup_grade"], base[prediction_columns(base)])["true"]}
    for head in HEADS:
        for loss in ["cross_entropy", *NEW_LOSSES]:
            run = table(run_name(head, loss), protocol).loc[base.index]
            shares[f"{head} {loss}"] = grade_distribution(run["isup_grade"], run[prediction_columns(run)])["predicted"]
    print(f"\n{protocol}: share of slides per grade")
    print(pd.DataFrame(shares).T.round(3).to_string())


radboud_to_karolinska: share of slides per grade
grade                       0      1      2      3      4      5
true                    0.353  0.333  0.122  0.058  0.088  0.046
abmil cross_entropy     0.386  0.253  0.055  0.162  0.094  0.051
abmil soft_labels       0.344  0.243  0.135  0.137  0.120  0.021
abmil corn              0.389  0.259  0.073  0.169  0.089  0.021
roformer cross_entropy  0.428  0.254  0.054  0.131  0.098  0.036
roformer soft_labels    0.405  0.214  0.126  0.149  0.079  0.028
roformer corn           0.426  0.271  0.043  0.120  0.106  0.034

karolinska_to_radboud: share of slides per grade
grade                       0      1      2      3      4      5
true                    0.187  0.165  0.131  0.179  0.149  0.189
abmil cross_entropy     0.207  0.151  0.139  0.096  0.296  0.112
abmil soft_labels       0.172  0.153  0.141  0.146  0.291  0.098
abmil corn              0.211  0.145  0.154  0.109  0.255  0.124
roformer cross_entropy  0.186  0.114  0.175  0.132  0.2

## The decision rule

Per head and loss, against the same head's cross-entropy: better in both
cross-site directions (interval above zero), not worse on mixed (interval not
below zero).

In [7]:
"""Apply the decision rule to each new loss within each head."""

for head in HEADS:
    for loss in NEW_LOSSES:
        rows = comparison[(comparison["head"] == head) & (comparison["loss"] == loss)
                          & (comparison["slides"] == "all")].set_index("protocol")
        cross_ok = all(rows.loc[p, "95% low"] > 0 for p in CROSS_SITE)
        mixed_ok = rows.loc["mixed", "95% high"] >= 0
        details = "; ".join(f"{p} {rows.loc[p, 'difference']:+.3f} [{rows.loc[p, '95% low']:+.3f}, "
                            f"{rows.loc[p, '95% high']:+.3f}]" for p in PROTOCOLS)
        print(f"{head:9s} {loss:12s} {'helps' if cross_ok and mixed_ok else 'does not help':14s} {details}")

abmil     soft_labels  helps          mixed +0.021 [+0.015, +0.027]; radboud_to_karolinska +0.065 [+0.050, +0.080]; karolinska_to_radboud +0.009 [+0.002, +0.015]
abmil     corn         helps          mixed +0.007 [+0.001, +0.013]; radboud_to_karolinska +0.081 [+0.067, +0.095]; karolinska_to_radboud +0.010 [+0.004, +0.017]
roformer  soft_labels  helps          mixed +0.013 [+0.007, +0.019]; radboud_to_karolinska +0.031 [+0.022, +0.041]; karolinska_to_radboud +0.028 [+0.022, +0.034]
roformer  corn         helps          mixed +0.002 [-0.004, +0.009]; radboud_to_karolinska +0.014 [+0.006, +0.023]; karolinska_to_radboud +0.011 [+0.006, +0.017]


## Result

Run 2026-11-18 to 19 on the RTX 4060 PC, packed tissue set
`uni2h-L1-224-t144-hsv-v1`; 27 minutes per loss for ABMIL, about 100 for the
RoPE model.

| Head | Loss | Mixed | Radboud to Karolinska (5 seeds) | Karolinska to Radboud (5 seeds) |
|---|---|---|---|---|
| ABMIL | cross-entropy | 0.864 | 0.632 (0.602 to 0.656) | 0.780 (0.775 to 0.784) |
| ABMIL | soft labels | 0.886 | 0.696 (0.678 to 0.709) | 0.789 (0.785 to 0.793) |
| ABMIL | CORN | 0.872 | 0.713 (0.702 to 0.723) | 0.791 (0.789 to 0.792) |
| RoPE model | cross-entropy | 0.874 | 0.719 (0.653 to 0.744) | 0.766 (0.753 to 0.777) |
| RoPE model | soft labels | **0.887** | **0.750** (0.722 to 0.780) | **0.794** (0.788 to 0.801) |
| RoPE model | CORN | 0.876 | 0.733 (0.708 to 0.748) | 0.777 (0.759 to 0.791) |

Each new loss against the same head's cross-entropy (95% paired bootstrap):

| Head | Loss | Mixed | Radboud to Karolinska | Karolinska to Radboud | Rule |
|---|---|---|---|---|---|
| ABMIL | soft labels | +0.021 [+0.015, +0.027] | +0.065 [+0.050, +0.080] | +0.009 [+0.002, +0.015] | helps |
| ABMIL | CORN | +0.007 [+0.001, +0.013] | +0.081 [+0.067, +0.095] | +0.010 [+0.004, +0.017] | helps |
| RoPE model | soft labels | +0.013 [+0.007, +0.019] | +0.031 [+0.022, +0.041] | +0.028 [+0.022, +0.034] | helps |
| RoPE model | CORN | +0.002 [-0.004, +0.009] | +0.014 [+0.006, +0.023] | +0.011 [+0.006, +0.017] | helps |

Heads compared at their best loss (computed after the run, same method): RoPE
model with soft labels minus ABMIL with soft labels: mixed +0.001 [-0.004,
+0.006], Radboud to Karolinska +0.054 [+0.042, +0.066], Karolinska to Radboud
+0.005 [+0.000, +0.010]. That passes the decision rule.

- **Both ordinal losses beat cross-entropy for both heads, in every
  direction.** Soft labels give the larger gain on mixed; CORN the larger gain
  from Radboud to Karolinska for ABMIL. This agrees with Abdul Haleem's ranking
  on EfficientNet (soft labels first).
- **The best combination is the RoPE model with soft labels**, highest in all
  three columns, and with soft labels it beats ABMIL in both cross-site
  directions, so the one-direction tie of notebook 16 is gone.
- Soft labels also cut the cross-site seed spread for ABMIL (Radboud to
  Karolinska std 0.020 to 0.013).
- Predicted grades: soft labels restore grade 2 from Radboud to Karolinska
  (ABMIL 5.5% to 13.5%, true 12.2%) and grade 3 the other way (9.6% to
  14.6%, true 17.9%); the excess of grade 4 from Karolinska to Radboud (about
  29%, true 14.9%) remains for ABMIL and is about 24% for the RoPE model.